# Analysis Pipeline

In this notebook we analyse the quality of the ScArches mapping using the *MappingAccessment* package. 

In [ ]:
# import os
# os.chdir('../')
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

In [ ]:
import scanpy as sc
import anndata
import matplotlib.pyplot as plt
import numpy as np
import scipy

import pandas as pd
import datetime
import statsmodels.stats.multitest as mtc
import json
import timeit
from MappingAssessment import MapAnalyzer, Plotter


In [ ]:
workspace = "C:\\Users\\49176\\PycharmProjects\\Bachelor\\HLCA"

In [ ]:
#set parameters here
dataset = "Meyer_2021_5_prime"
ident = "Meyer_2021_5_prime"
query_color = "Blue"
condition = "X"
exclude = []

In [ ]:
adata= sc.read(f'{workspace}\\Adatas\\HLCA_full_v1.h5ad')

In [ ]:
adata = adata[(adata.obs.core_or_extension=="core")|((adata.obs.dataset==dataset))]

### Preprocessing

Set clean cell labels on all levels and one label-column for transfer from curated metadata, assumed to be done in the package 

In [ ]:
# In case that no umap based on the embedding is calculated, recalculate the UMAP
# sc.pp.neighbors(adata, n_neighbors=50, n_pcs=30)
# sc.tl.umap(adata)

Initialize *AdataHandler* object. The *AdataHandler* makes sure all columns are named a standarized way. It also is able to perform a label transfer to the query.

In [ ]:
import imp

imp.reload(MapAnalyzer)

In [ ]:
adata_handler = MapAnalyzer.AdataHandler(
    adata,
    workspace=f"{workspace}\\MapAnalyzer_Results_3.0\\",
    ident=ident+"\\",  # folder that indentifies this workpartition
    log=True,  # create a logfile
)

In [ ]:
# Plotter object for the plots
plotter = Plotter.MapPlot(workspace=f"{adata_handler.workspace}/{ident}")

In [ ]:
adata_handler.standarize_columns(
    condition_name=condition,  # condition name we want to distinguish
    query_dataset=dataset,  # query datasets
    cell_type_col=None,  # column name denoting celltype, none if no unified column
    dataset_col="dataset",  # column name denoting dataset
    sample_col="sample",  # column name danoting sample
    condition_col="lung_condition",  # column name denoting condition (abovementioned)
    subject_col="donor_id",  # column name denoting patients, subjects or donors
    disease_exclude=exclude,  # diseases to exclude from analysis
)

Set appropriate levels for the cell type annoation

In [ ]:
adata_handler.set_cell_type_levels(1,['ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5'],"ann_finest_level","nan")

In [ ]:
adata= adata_handler.get_adata()

Transfer celltype labels from reference data to query cells

In [ ]:
adata_handler.transfer_labels(transfer_columns=["cell_label"],pickle=False)

In [ ]:
sc.pl.umap(adata,color="cell_label")

In [ ]:
adata.obs[["query_or_reference","cell_label"]]

In [ ]:
plotter.umap_query_reference(
    adata, color_query=query_color, title_stub=ident, save=False
)

In [ ]:
import imp

imp.reload(MapAnalyzer)

### Main Pipeline

Initialize the *MappingScoring* object. Pass the *AdataHandler* so that adata and workspace information is passed on.

In [ ]:
scorer = MapAnalyzer.MappingScoring(adata_handler)

In [ ]:
adata = scorer.get_adata()

#### Calculate or pass query mixing scores

In [ ]:
'''scorer.calculate_mixing_scores(
    n_neighbors=15,
    min_subjects=40,
    samples=None,
    reinitialize=True,
    embedding="X",
    verbose=True,
    overwrite=True,
    save=True,
)'''

In [ ]:
scorer.add_scores(
    f"{scorer.workspace}\\{ident}\\scores.csv",
)

**Baseline calculation:**  
To compare our mapping to a upper bound, we calculate the scores for the integrated reference. For that we set one sample to query, while the other samples are set to reference (excluding the real query). By that we achieve a baseline for each celltype of how well different samples are integrated. We then can compare out mapping scores to the baseline to see, wether the mapping is of comparable quality to the integration. 

In [ ]:
#scorer.calculate_baseline(save=True,verbose=False)

In [ ]:
scorer.add_baseline(
    f"{scorer.workspace}\\baseline.csv",
    f"{workspace}\\MapAnalyzer_Results_3.0\\celltypes.txt",
)

add correction model for score correction for sample size:

In [ ]:
scorer.add_transformation(f"{workspace}\\MapAnalyzer_Results_3.0\\transformation.csv")

### Analyzing the Mapping with MapAnalyzer

In [ ]:
adata = scorer.get_adata()

In [ ]:
scorer.analyze_mapping()

In [ ]:
scorer.sample_report

In [ ]:
scorer.report.to_csv(
    f"{workspace}\\MapAnalyzer_Results_3.0\\{ident}\\report.csv"
)

### Plotting the outcomes

In [ ]:
report = scorer.report.loc

In [ ]:
adata = adata[adata.obs.cell_label.isin(scorer.cell_types)]

In [ ]:
adata

In [ ]:
plotter.plot_baseline_comparison(
    adata,
    query_color,
    ident,
    disease=True,
    save=True,
    violin=True,
    #sample_filter=report,
    transform=f"{workspace}\\MapAnalyzer_Results_3.0\\transformation.csv",
)

In [ ]:
## New threshold based on transformed scores

In [ ]:
def group_adata(adata):
    group_adata = (
        adata.obs.groupby(["cell_label", "sample"])
        .agg({"score": "mean", "dataset": "first", "subject_ID": "count"})
        .dropna()
        .reset_index(drop=False)
    )

    group_adata["cell_label"] = group_adata["cell_label"].cat.remove_unused_categories()
    group_adata = group_adata.rename(columns={"subject_ID": "count"})
    group_adata["log_count"] = np.log(group_adata["count"])
    return group_adata